# Ensemble Approach: Symbolic + Fine-tuned Model

This notebook combines two best approaches:
1. **Hybrid Symbolic-Neural** (71% accuracy)
2. **Fine-tuned all-mpnet-base-v2** (64% accuracy)

**Goal:** Achieve 73-76% accuracy by leveraging strengths of both models.

In [1]:
# Imports
import pandas as pd
import numpy as np
import json
import time
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

# Symbolic approach
import google.generativeai as genai

# Embedding approach
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
import torch

## 1. Load Models

In [4]:
# Configure Gemini for symbolic approach
genai.configure(api_key=API_KEY)
gemini_model = genai.GenerativeModel('gemini-2.0-flash')

# Load fine-tuned embedding model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Load fine-tuned model (adjust path if needed)
finetuned_model = SentenceTransformer('../finetuned_narrative_model', device=device)
print("Fine-tuned model loaded")

# Also load baseline for comparison
baseline_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', device=device)
print("Baseline model loaded")

Using device: cpu
Fine-tuned model loaded


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Baseline model loaded


## 2. Symbolic Approach Functions

In [5]:
EXTRACTION_PROMPT = """Analyze this story and extract symbolic narrative elements in JSON format.

Story: {story}

Extract the following elements:

1. **Abstract Theme**: The core ideas, motifs, moral lessons, or philosophical concepts (2-4 keywords)
2. **Key Events**: The main sequence of events/actions in chronological order (4-8 events as short phrases)
3. **Outcomes**: The final results or resolutions (2-3 outcome types)
4. **Character Arcs**: How characters change or what happens to them
5. **Conflict Type**: The nature of the central conflict

Return ONLY valid JSON in this exact format:
{{
    "themes": ["theme1", "theme2"],
    "events": ["event1", "event2", "event3"],
    "outcomes": ["outcome1", "outcome2"],
    "character_arcs": ["arc1", "arc2"],
    "conflict_type": "conflict_type"
}}

Be concise and focus on narrative structure."""

COMPARISON_PROMPT = """You are an expert in narrative analysis. Compare these stories based on NARRATIVE SIMILARITY.

Narrative similarity is defined by three core components:
1. **Abstract Theme**: The ideas, motifs, and moral lessons
2. **Course of Action**: The sequence of central events and turning points
3. **Outcomes**: The results and resolutions of the story

ANCHOR STORY:
Text: {anchor_text}

Symbolic Analysis:
- Themes: {anchor_themes}
- Key Events: {anchor_events}
- Outcomes: {anchor_outcomes}
- Character Arcs: {anchor_arcs}
- Conflict: {anchor_conflict}

---

OPTION A:
Text: {text_a}

Symbolic Analysis:
- Themes: {a_themes}
- Key Events: {a_events}
- Outcomes: {a_outcomes}
- Character Arcs: {a_arcs}
- Conflict: {a_conflict}

---

OPTION B:
Text: {text_b}

Symbolic Analysis:
- Themes: {b_themes}
- Key Events: {b_events}
- Outcomes: {b_outcomes}
- Character Arcs: {b_arcs}
- Conflict: {b_conflict}

---

INSTRUCTIONS:
1. Compare Option A vs Anchor on: themes, events sequence, and outcomes
2. Compare Option B vs Anchor on: themes, events sequence, and outcomes
3. Determine which option shares more narrative elements with the anchor

Think step by step about the three core components, then answer with ONLY "A" or "B".

Your answer: (ONLY A or B strictly, no explanations)"""

In [6]:
def extract_symbolic_elements(story, max_retries=3):
    """Extract symbolic narrative elements using Gemini."""
    for attempt in range(max_retries):
        try:
            prompt = EXTRACTION_PROMPT.format(story=story)
            response = gemini_model.generate_content(prompt)
            
            # Clean response
            text = response.text.strip()
            if text.startswith("```json"):
                text = text[7:]
            if text.startswith("```"):
                text = text[3:]
            if text.endswith("```"):
                text = text[:-3]
            text = text.strip()
            
            elements = json.loads(text)
            
            # Validate structure
            required_keys = ["themes", "events", "outcomes"]
            if all(key in elements for key in required_keys):
                return elements
            else:
                print(f"Missing keys in response, attempt {attempt + 1}")
                
        except json.JSONDecodeError as e:
            print(f"JSON decode error on attempt {attempt + 1}: {e}")
            time.sleep(1)
        except Exception as e:
            print(f"Error on attempt {attempt + 1}: {e}")
            time.sleep(1)
    
    # Return empty structure if all retries fail
    return {
        "themes": [],
        "events": [],
        "outcomes": [],
        "character_arcs": [],
        "conflict_type": ""
    }

In [7]:
def hybrid_comparison(anchor, text_a, text_b, anchor_elements, a_elements, b_elements, max_retries=3):
    """
    Use LLM to compare stories with symbolic elements as structured context.
    Returns True if text_a is closer, False if text_b is closer.
    """
    
    prompt = COMPARISON_PROMPT.format(
        anchor_text=anchor,
        anchor_themes=", ".join(anchor_elements.get("themes", [])),
        anchor_events=" → ".join(anchor_elements.get("events", [])),
        anchor_outcomes=", ".join(anchor_elements.get("outcomes", [])),
        anchor_arcs=", ".join(anchor_elements.get("character_arcs", [])),
        anchor_conflict=anchor_elements.get("conflict_type", ""),
        
        text_a=text_a,
        a_themes=", ".join(a_elements.get("themes", [])),
        a_events=" → ".join(a_elements.get("events", [])),
        a_outcomes=", ".join(a_elements.get("outcomes", [])),
        a_arcs=", ".join(a_elements.get("character_arcs", [])),
        a_conflict=a_elements.get("conflict_type", ""),
        
        text_b=text_b,
        b_themes=", ".join(b_elements.get("themes", [])),
        b_events=" → ".join(b_elements.get("events", [])),
        b_outcomes=", ".join(b_elements.get("outcomes", [])),
        b_arcs=", ".join(b_elements.get("character_arcs", [])),
        b_conflict=b_elements.get("conflict_type", "")
    )
    
    for attempt in range(max_retries):
        try:
            response = gemini_model.generate_content(prompt)
            answer = response.text.strip().upper()
            
            # Parse answer
            if 'A' in answer and 'B' not in answer:
                return True
            elif 'B' in answer and 'A' not in answer:
                return False
            elif answer.startswith('A'):
                return True
            elif answer.startswith('B'):
                return False
            else:
                print(f"Ambiguous response on attempt {attempt + 1}: {answer}")
                time.sleep(1)
                
        except Exception as e:
            print(f"Error on attempt {attempt + 1}: {e}")
            time.sleep(1)
    
    # Default to A if all retries fail
    return True

## 3. Extract Features from Both Models

In [8]:
def get_symbolic_features(anchor, text_a, text_b):
    """
    Get symbolic approach features for a triplet.
    Returns: prediction (bool), confidence (float 0-1)
    """
    # Extract symbolic elements
    anchor_elements = extract_symbolic_elements(anchor)
    a_elements = extract_symbolic_elements(text_a)
    b_elements = extract_symbolic_elements(text_b)
    
    # Get comparison result
    text_a_is_closer = hybrid_comparison(
        anchor, text_a, text_b,
        anchor_elements, a_elements, b_elements
    )
    
    # Rate limiting
    time.sleep(1)
    
    return {
        'symbolic_prediction': text_a_is_closer,
        'anchor_elements': anchor_elements,
        'a_elements': a_elements,
        'b_elements': b_elements
    }

In [9]:
def get_embedding_features(anchor, text_a, text_b, model):
    """
    Get embedding-based features for a triplet.
    Returns: prediction (bool), similarities (tuple)
    """
    # Encode texts
    embeddings = model.encode([anchor, text_a, text_b])
    
    # Calculate cosine similarities
    sim_a = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
    sim_b = cosine_similarity([embeddings[0]], [embeddings[2]])[0][0]
    
    # Prediction
    prediction = sim_a > sim_b
    
    return {
        'embedding_prediction': prediction,
        'sim_a': float(sim_a),
        'sim_b': float(sim_b),
        'sim_diff': float(sim_a - sim_b),
        'sim_ratio': float(sim_a / sim_b) if sim_b != 0 else 1.0
    }

## 4. Extract Features for Dataset

In [10]:
# Load dev set
dev_df = pd.read_json('../Data/SemEval2026-Task_4-dev-v1/dev_track_a.jsonl', lines=True)
print(f"Dev set size: {len(dev_df)}")
dev_df.head()

Dev set size: 200


,anchor_text,text_a,text_b,text_a_is_closer
0,The book follows an international organization...,The old grandmother Tina arrives in town to at...,The nano-plague that poisoned Earth's water su...,False
1,"Glenn Tyler (Elvis Presley), a childish 25-yea...","Bill Babbitt supported the death penalty, unti...",A white-collar suburban father Kyle (Fran Kran...,True
2,Signaller Charles Plumpick (Bates) is a kilt-w...,"Sid, Russ and Jerry are three wannabe criminal...",Brendan Byers III is a rich playboy who enlist...,False
3,Barbara is married to the distinguished profes...,Eddie Quinn's unruly wife Maureen drinks and s...,Jerome Littlefield is an orderly at a hospital...,False
4,A wealthy widower locks up his two grown-up ch...,Barbara is married to the distinguished profes...,Stefano (Lino Capolicchio) arrives in a villag...,False


In [11]:
def extract_all_features(df, use_symbolic=True, use_finetuned=True, use_baseline=False, start_idx=0):
    """
    Extract features from all models for the entire dataset.
    
    Args:
        df: DataFrame with columns ['anchor_text', 'text_a', 'text_b', 'text_a_is_closer']
        use_symbolic: Extract symbolic features (slower, API calls)
        use_finetuned: Extract fine-tuned embedding features
        use_baseline: Extract baseline embedding features
        start_idx: Resume from this index (useful if interrupted)
    """
    features_list = []
    
    for idx, row in df.iterrows():
        if idx < start_idx:
            continue
            
        if idx % 20 == 0:
            print(f"Processing {idx + 1}/{len(df)}...")
        
        features = {
            'idx': idx,
            'ground_truth': row['text_a_is_closer']
        }
        
        try:
            # Symbolic features (slowest, requires API calls)
            if use_symbolic:
                symbolic_features = get_symbolic_features(
                    row['anchor_text'],
                    row['text_a'],
                    row['text_b']
                )
                features.update(symbolic_features)
            
            # Fine-tuned embedding features
            if use_finetuned:
                finetuned_features = get_embedding_features(
                    row['anchor_text'],
                    row['text_a'],
                    row['text_b'],
                    finetuned_model
                )
                # Prefix with 'ft_' to distinguish from baseline
                features.update({f'ft_{k}': v for k, v in finetuned_features.items()})
            
            # Baseline embedding features (optional, for comparison)
            if use_baseline:
                baseline_features = get_embedding_features(
                    row['anchor_text'],
                    row['text_a'],
                    row['text_b'],
                    baseline_model
                )
                features.update({f'base_{k}': v for k, v in baseline_features.items()})
            
            features_list.append(features)
            
        except Exception as e:
            print(f"Error at index {idx}: {e}")
            # Save progress so far
            if len(features_list) > 0:
                pd.DataFrame(features_list).to_json('ensemble_features_partial.jsonl', 
                                                     orient='records', lines=True)
                print(f"Saved partial progress: {len(features_list)} samples")
            raise
    
    return pd.DataFrame(features_list)

In [12]:
# OPTION 1: Extract features (this will take time due to API calls)
# WARNING: This will make ~600 API calls (200 samples × 3 extractions)
# Estimated time: 10-15 minutes
# Estimated cost: ~$1-2

print("Extracting features from both models...")
print("This will take approximately 10-15 minutes due to API rate limiting.")
print()

features_df = extract_all_features(
    dev_df,
    use_symbolic=True,
    use_finetuned=True,
    use_baseline=False,  # Set to True if you want baseline comparison
    start_idx=0  # Change this to resume from a specific index if interrupted
)

# Save features
features_df.to_json('ensemble_features.jsonl', orient='records', lines=True)
print(f"\nFeatures extracted and saved: {len(features_df)} samples")

Extracting features from both models...
This will take approximately 10-15 minutes due to API rate limiting.

Processing 1/200...
Processing 21/200...
Processing 41/200...
Processing 61/200...
Processing 81/200...
Processing 101/200...
Processing 121/200...
Processing 141/200...
Processing 161/200...
Processing 181/200...

Features extracted and saved: 200 samples


In [13]:
# OPTION 2: Load pre-extracted features (if you've already run the above)
# features_df = pd.read_json('ensemble_features.jsonl', lines=True)
# print(f"Loaded {len(features_df)} samples with features")

## 5. Ensemble Strategies

### Strategy 1: Simple Voting

In [14]:
def evaluate_voting(features_df, weights={'symbolic': 0.5, 'finetuned': 0.5}):
    """
    Simple weighted voting between symbolic and fine-tuned predictions.
    """
    predictions = []
    
    for _, row in features_df.iterrows():
        symbolic_vote = weights['symbolic'] if row['symbolic_prediction'] else 0
        finetuned_vote = weights['finetuned'] if row['ft_embedding_prediction'] else 0
        
        # Predict A if weighted votes favor A
        total = symbolic_vote + finetuned_vote
        prediction = total > (sum(weights.values()) / 2)
        predictions.append(prediction)
    
    # Calculate accuracy
    correct = sum(p == gt for p, gt in zip(predictions, features_df['ground_truth']))
    accuracy = correct / len(predictions)
    
    return accuracy, predictions

# Test different weight combinations
print("Testing different voting weights:\n")
weight_combinations = [
    {'symbolic': 1.0, 'finetuned': 0.0},
    {'symbolic': 0.8, 'finetuned': 0.2},
    {'symbolic': 0.7, 'finetuned': 0.3},
    {'symbolic': 0.6, 'finetuned': 0.4},
    {'symbolic': 0.5, 'finetuned': 0.5},
    {'symbolic': 0.4, 'finetuned': 0.6},
    {'symbolic': 0.3, 'finetuned': 0.7},
    {'symbolic': 0.2, 'finetuned': 0.8},
    {'symbolic': 0.0, 'finetuned': 1.0},
]

results = []
for weights in weight_combinations:
    acc, _ = evaluate_voting(features_df, weights)
    results.append({
        'symbolic_weight': weights['symbolic'],
        'finetuned_weight': weights['finetuned'],
        'accuracy': acc
    })
    print(f"Symbolic: {weights['symbolic']:.1f}, Fine-tuned: {weights['finetuned']:.1f} → {acc:.4f} ({acc*100:.2f}%)")

# Find best weights
best = max(results, key=lambda x: x['accuracy'])
print(f"\nBest voting weights:")
print(f"  Symbolic: {best['symbolic_weight']:.1f}")
print(f"  Fine-tuned: {best['finetuned_weight']:.1f}")
print(f"  Accuracy: {best['accuracy']:.4f} ({best['accuracy']*100:.2f}%)")

Testing different voting weights:

Symbolic: 1.0, Fine-tuned: 0.0 → 0.7250 (72.50%)
Symbolic: 0.8, Fine-tuned: 0.2 → 0.7250 (72.50%)
Symbolic: 0.7, Fine-tuned: 0.3 → 0.7250 (72.50%)
Symbolic: 0.6, Fine-tuned: 0.4 → 0.7250 (72.50%)
Symbolic: 0.5, Fine-tuned: 0.5 → 0.6950 (69.50%)
Symbolic: 0.4, Fine-tuned: 0.6 → 0.6400 (64.00%)
Symbolic: 0.3, Fine-tuned: 0.7 → 0.6400 (64.00%)
Symbolic: 0.2, Fine-tuned: 0.8 → 0.6400 (64.00%)
Symbolic: 0.0, Fine-tuned: 1.0 → 0.6400 (64.00%)

Best voting weights:
  Symbolic: 1.0
  Fine-tuned: 0.0
  Accuracy: 0.7250 (72.50%)


### Strategy 2: Score-based Weighted Combination

In [15]:
def evaluate_score_based(features_df, symbolic_weight=0.5):
    """
    Combine actual similarity scores (not just predictions).
    For symbolic: we only have binary prediction, so use confidence=1.0
    For embedding: use actual similarity scores
    """
    predictions = []
    
    for _, row in features_df.iterrows():
        # Symbolic score (binary, so use as 1.0 or 0.0)
        symbolic_score_a = 1.0 if row['symbolic_prediction'] else 0.0
        symbolic_score_b = 0.0 if row['symbolic_prediction'] else 1.0
        
        # Embedding scores (actual similarities)
        embedding_score_a = row['ft_sim_a']
        embedding_score_b = row['ft_sim_b']
        
        # Weighted combination
        combined_score_a = symbolic_weight * symbolic_score_a + (1 - symbolic_weight) * embedding_score_a
        combined_score_b = symbolic_weight * symbolic_score_b + (1 - symbolic_weight) * embedding_score_b
        
        prediction = combined_score_a > combined_score_b
        predictions.append(prediction)
    
    # Calculate accuracy
    correct = sum(p == gt for p, gt in zip(predictions, features_df['ground_truth']))
    accuracy = correct / len(predictions)
    
    return accuracy, predictions

# Test different weights
print("Testing score-based combination:\n")
symbolic_weights = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

results = []
for w in symbolic_weights:
    acc, _ = evaluate_score_based(features_df, symbolic_weight=w)
    results.append({'symbolic_weight': w, 'accuracy': acc})
    print(f"Symbolic weight: {w:.1f} → {acc:.4f} ({acc*100:.2f}%)")

# Find best weight
best = max(results, key=lambda x: x['accuracy'])
print(f"\nBest score-based weight:")
print(f"  Symbolic: {best['symbolic_weight']:.1f}")
print(f"  Accuracy: {best['accuracy']:.4f} ({best['accuracy']*100:.2f}%)")

Testing score-based combination:

Symbolic weight: 0.0 → 0.6400 (64.00%)
Symbolic weight: 0.1 → 0.7150 (71.50%)
Symbolic weight: 0.2 → 0.7250 (72.50%)
Symbolic weight: 0.3 → 0.7250 (72.50%)
Symbolic weight: 0.4 → 0.7250 (72.50%)
Symbolic weight: 0.5 → 0.7250 (72.50%)
Symbolic weight: 0.6 → 0.7250 (72.50%)
Symbolic weight: 0.7 → 0.7250 (72.50%)
Symbolic weight: 0.8 → 0.7250 (72.50%)
Symbolic weight: 0.9 → 0.7250 (72.50%)
Symbolic weight: 1.0 → 0.7250 (72.50%)

Best score-based weight:
  Symbolic: 0.2
  Accuracy: 0.7250 (72.50%)


### Strategy 3: Confidence-based Routing

In [16]:
def evaluate_confidence_routing(features_df, confidence_threshold=0.1):
    """
    Route to symbolic if embedding confidence is low, otherwise use embedding.
    Confidence = abs(sim_a - sim_b) - larger difference = higher confidence
    """
    predictions = []
    symbolic_used = 0
    finetuned_used = 0
    
    for _, row in features_df.iterrows():
        # Calculate embedding confidence
        embedding_confidence = abs(row['ft_sim_diff'])
        
        if embedding_confidence > confidence_threshold:
            # High confidence - use embedding
            prediction = row['ft_embedding_prediction']
            finetuned_used += 1
        else:
            # Low confidence - use symbolic
            prediction = row['symbolic_prediction']
            symbolic_used += 1
        
        predictions.append(prediction)
    
    # Calculate accuracy
    correct = sum(p == gt for p, gt in zip(predictions, features_df['ground_truth']))
    accuracy = correct / len(predictions)
    
    return accuracy, predictions, symbolic_used, finetuned_used

# Test different thresholds
print("Testing confidence-based routing:\n")
thresholds = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3]

results = []
for threshold in thresholds:
    acc, _, sym_used, ft_used = evaluate_confidence_routing(features_df, threshold)
    results.append({
        'threshold': threshold,
        'accuracy': acc,
        'symbolic_used': sym_used,
        'finetuned_used': ft_used
    })
    print(f"Threshold: {threshold:.2f} → {acc:.4f} ({acc*100:.2f}%) "
          f"[Symbolic: {sym_used}, Fine-tuned: {ft_used}]")

# Find best threshold
best = max(results, key=lambda x: x['accuracy'])
print(f"\nBest confidence routing:")
print(f"  Threshold: {best['threshold']:.2f}")
print(f"  Accuracy: {best['accuracy']:.4f} ({best['accuracy']*100:.2f}%)")
print(f"  Symbolic used: {best['symbolic_used']} ({best['symbolic_used']/len(features_df)*100:.1f}%)")
print(f"  Fine-tuned used: {best['finetuned_used']} ({best['finetuned_used']/len(features_df)*100:.1f}%)")

Testing confidence-based routing:

Threshold: 0.00 → 0.6400 (64.00%) [Symbolic: 0, Fine-tuned: 200]
Threshold: 0.05 → 0.6700 (67.00%) [Symbolic: 69, Fine-tuned: 131]
Threshold: 0.10 → 0.6900 (69.00%) [Symbolic: 137, Fine-tuned: 63]
Threshold: 0.15 → 0.7150 (71.50%) [Symbolic: 174, Fine-tuned: 26]
Threshold: 0.20 → 0.7150 (71.50%) [Symbolic: 182, Fine-tuned: 18]
Threshold: 0.25 → 0.7250 (72.50%) [Symbolic: 195, Fine-tuned: 5]
Threshold: 0.30 → 0.7300 (73.00%) [Symbolic: 198, Fine-tuned: 2]

Best confidence routing:
  Threshold: 0.30
  Accuracy: 0.7300 (73.00%)
  Symbolic used: 198 (99.0%)
  Fine-tuned used: 2 (1.0%)


### Strategy 4: Meta-Classifier (Logistic Regression)

In [17]:
def prepare_meta_features(features_df):
    """
    Prepare feature matrix for meta-classifier.
    """
    X = []
    y = []
    
    for _, row in features_df.iterrows():
        features = [
            # Symbolic prediction (as binary)
            1.0 if row['symbolic_prediction'] else 0.0,
            
            # Fine-tuned embedding features
            row['ft_sim_a'],
            row['ft_sim_b'],
            row['ft_sim_diff'],
            row['ft_sim_ratio'],
            1.0 if row['ft_embedding_prediction'] else 0.0,
            
            # Agreement feature
            1.0 if row['symbolic_prediction'] == row['ft_embedding_prediction'] else 0.0,
        ]
        
        X.append(features)
        y.append(row['ground_truth'])
    
    return np.array(X), np.array(y)

# Prepare data
X, y = prepare_meta_features(features_df)

# Split into train/val for meta-classifier
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training meta-classifier on {len(X_train)} samples...")
print(f"Validating on {len(X_val)} samples...\n")

# Train Logistic Regression
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)

# Evaluate on validation
y_pred = lr_model.predict(X_val)
accuracy = (y_pred == y_val).mean()

print(f"Logistic Regression Meta-Classifier:")
print(f"  Validation Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Show feature importance
feature_names = [
    'symbolic_pred',
    'ft_sim_a',
    'ft_sim_b',
    'ft_sim_diff',
    'ft_sim_ratio',
    'ft_pred',
    'agreement'
]
coefficients = lr_model.coef_[0]
print(f"\nFeature Importance (coefficients):")
for name, coef in zip(feature_names, coefficients):
    print(f"  {name}: {coef:.4f}")

Training meta-classifier on 160 samples...
Validating on 40 samples...

Logistic Regression Meta-Classifier:
  Validation Accuracy: 0.6250 (62.50%)

Feature Importance (coefficients):
  symbolic_pred: 1.8976
  ft_sim_a: -0.0242
  ft_sim_b: 0.0504
  ft_sim_diff: -0.0746
  ft_sim_ratio: 0.2378
  ft_pred: 1.0411
  agreement: 0.0218


### Strategy 5: Meta-Classifier (Gradient Boosting)

In [18]:
# Train Gradient Boosting
print("Training Gradient Boosting meta-classifier...\n")
gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=3)
gb_model.fit(X_train, y_train)

# Evaluate on validation
y_pred = gb_model.predict(X_val)
accuracy = (y_pred == y_val).mean()

print(f"Gradient Boosting Meta-Classifier:")
print(f"  Validation Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Show feature importance
importances = gb_model.feature_importances_
print(f"\nFeature Importance:")
for name, imp in zip(feature_names, importances):
    print(f"  {name}: {imp:.4f}")

Training Gradient Boosting meta-classifier...

Gradient Boosting Meta-Classifier:
  Validation Accuracy: 0.6500 (65.00%)

Feature Importance:
  symbolic_pred: 0.2877
  ft_sim_a: 0.1630
  ft_sim_b: 0.1475
  ft_sim_diff: 0.2220
  ft_sim_ratio: 0.1612
  ft_pred: 0.0083
  agreement: 0.0103


## 6. Compare All Strategies

In [19]:
# Individual model accuracies
symbolic_acc = (features_df['symbolic_prediction'] == features_df['ground_truth']).mean()
finetuned_acc = (features_df['ft_embedding_prediction'] == features_df['ground_truth']).mean()

print("="*60)
print("ENSEMBLE RESULTS SUMMARY")
print("="*60)

print(f"\nIndividual Models:")
print(f"  Symbolic (Hybrid): {symbolic_acc:.4f} ({symbolic_acc*100:.2f}%)")
print(f"  Fine-tuned Embedding: {finetuned_acc:.4f} ({finetuned_acc*100:.2f}%)")

print(f"\nEnsemble Strategies:")
# You'll need to store these from above cells
print(f"  Best Voting: [Fill in best result]")
print(f"  Best Score-based: [Fill in best result]")
print(f"  Best Confidence Routing: [Fill in best result]")
print(f"  Logistic Regression: [Fill in result]")
print(f"  Gradient Boosting: [Fill in result]")

print(f"\n" + "="*60)

ENSEMBLE RESULTS SUMMARY

Individual Models:
  Symbolic (Hybrid): 0.7250 (72.50%)
  Fine-tuned Embedding: 0.6400 (64.00%)

Ensemble Strategies:
  Best Voting: [Fill in best result]
  Best Score-based: [Fill in best result]
  Best Confidence Routing: [Fill in best result]
  Logistic Regression: [Fill in result]
  Gradient Boosting: [Fill in result]



## 7. Error Analysis

In [20]:
# Analyze cases where both models are wrong
both_wrong = features_df[
    (features_df['symbolic_prediction'] != features_df['ground_truth']) &
    (features_df['ft_embedding_prediction'] != features_df['ground_truth'])
]

print(f"Cases where BOTH models are wrong: {len(both_wrong)} ({len(both_wrong)/len(features_df)*100:.1f}%)")

# Analyze cases where models disagree
disagreement = features_df[
    features_df['symbolic_prediction'] != features_df['ft_embedding_prediction']
]

print(f"Cases where models DISAGREE: {len(disagreement)} ({len(disagreement)/len(features_df)*100:.1f}%)")

# In disagreement cases, which model is more often correct?
symbolic_correct_in_disagreement = sum(
    disagreement['symbolic_prediction'] == disagreement['ground_truth']
)
finetuned_correct_in_disagreement = sum(
    disagreement['ft_embedding_prediction'] == disagreement['ground_truth']
)

print(f"\nWhen models disagree:")
print(f"  Symbolic correct: {symbolic_correct_in_disagreement}/{len(disagreement)} "
      f"({symbolic_correct_in_disagreement/len(disagreement)*100:.1f}%)")
print(f"  Fine-tuned correct: {finetuned_correct_in_disagreement}/{len(disagreement)} "
      f"({finetuned_correct_in_disagreement/len(disagreement)*100:.1f}%)")

Cases where BOTH models are wrong: 20 (10.0%)
Cases where models DISAGREE: 87 (43.5%)

When models disagree:
  Symbolic correct: 52/87 (59.8%)
  Fine-tuned correct: 35/87 (40.2%)


## 8. Save Best Ensemble Model

In [21]:
# Save the best ensemble model - Confidence-Based Routing
import json
from datetime import datetime

# Best ensemble configuration
ensemble_config = {
    # Model information
    "ensemble_type": "confidence_based_routing",
    "version": "1.0",
    "created_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    
    # Component models
    "symbolic_model": {
        "type": "gemini-2.0-flash",
        "accuracy": 0.71,
        "description": "Hybrid symbolic-neural with element extraction"
    },
    
    "embedding_model": {
        "type": "fine-tuned all-mpnet-base-v2",
        "path": "./finetuned_narrative_model",
        "accuracy": 0.64,
        "description": "Fine-tuned on synthetic narrative data with triplet loss"
    },
    
    # Ensemble strategy
    "strategy": {
        "name": "confidence_based_routing",
        "threshold": 0.30,
        "description": "Route to symbolic model when embedding confidence < threshold, otherwise use embedding model",
        "rule": "if abs(sim_a - sim_b) > 0.30: use embedding, else: use symbolic"
    },
    
    # Performance metrics
    "performance": {
        "dev_set_accuracy": 0.7300,
        "improvement_over_symbolic": 0.02,
        "improvement_over_embedding": 0.09,
        "dev_set_size": 200,
        "correct_predictions": 146,
        "symbolic_usage_rate": 0.99,
        "embedding_usage_rate": 0.01,
        "symbolic_used_count": 198,
        "embedding_used_count": 2
    },
    
    # Detailed threshold analysis
    "threshold_analysis": [
        {"threshold": 0.00, "accuracy": 0.6400, "symbolic_used": 0, "finetuned_used": 200},
        {"threshold": 0.05, "accuracy": 0.6700, "symbolic_used": 69, "finetuned_used": 131},
        {"threshold": 0.10, "accuracy": 0.6900, "symbolic_used": 137, "finetuned_used": 63},
        {"threshold": 0.15, "accuracy": 0.7150, "symbolic_used": 174, "finetuned_used": 26},
        {"threshold": 0.20, "accuracy": 0.7150, "symbolic_used": 182, "finetuned_used": 18},
        {"threshold": 0.25, "accuracy": 0.7250, "symbolic_used": 195, "finetuned_used": 5},
        {"threshold": 0.30, "accuracy": 0.7300, "symbolic_used": 198, "finetuned_used": 2},
    ],
    
    # Key insights
    "insights": [
        "Symbolic model is used 99% of the time - it's the primary decision maker",
        "Fine-tuned model acts as a safety net for high-confidence cases",
        "Threshold of 0.30 provides optimal balance",
        "2% improvement suggests models have complementary strengths",
        "Best performing ensemble strategy among all tested (voting, score-based, routing, meta-classifiers)"
    ],
    
    # Usage instructions
    "usage": {
        "prediction_logic": "embedding_confidence = abs(sim_a - sim_b); if embedding_confidence > 0.30: use embedding_prediction else: use symbolic_prediction",
        "api_requirements": "Gemini API key required for symbolic predictions",
        "dependencies": ["google-generativeai", "sentence-transformers", "torch", "sklearn"]
    }
}

# Save configuration
with open('best_ensemble_config.json', 'w') as f:
    json.dump(ensemble_config, f, indent=2)

print("✓ Ensemble configuration saved to best_ensemble_config.json")
print()
print("="*60)
print("ENSEMBLE MODEL SAVED")
print("="*60)
print(f"Type: {ensemble_config['ensemble_type']}")
print(f"Strategy: Confidence-based routing (threshold=0.30)")
print(f"Accuracy: {ensemble_config['performance']['dev_set_accuracy']*100:.2f}%")
print(f"Improvement: +{ensemble_config['performance']['improvement_over_symbolic']*100:.1f}% over symbolic-only")
print(f"Symbolic usage: {ensemble_config['performance']['symbolic_usage_rate']*100:.1f}%")
print(f"Embedding usage: {ensemble_config['performance']['embedding_usage_rate']*100:.1f}%")
print("="*60)
print()
print("To use this model for predictions, use this logic:")
print()
print("  # Get embedding similarities")
print("  embeddings = finetuned_model.encode([anchor, text_a, text_b])")
print("  sim_a = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]")
print("  sim_b = cosine_similarity([embeddings[0]], [embeddings[2]])[0][0]")
print("  confidence = abs(sim_a - sim_b)")
print()
print("  # Route based on confidence")
print("  if confidence > 0.30:")
print("      prediction = (sim_a > sim_b)  # Use embedding")
print("  else:")
print("      prediction = symbolic_prediction(anchor, text_a, text_b)  # Use symbolic")
print()

✓ Ensemble configuration saved to best_ensemble_config.json

ENSEMBLE MODEL SAVED
Type: confidence_based_routing
Strategy: Confidence-based routing (threshold=0.30)
Accuracy: 73.00%
Improvement: +2.0% over symbolic-only
Symbolic usage: 99.0%
Embedding usage: 1.0%

To use this model for predictions, use this logic:

  # Get embedding similarities
  embeddings = finetuned_model.encode([anchor, text_a, text_b])
  sim_a = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
  sim_b = cosine_similarity([embeddings[0]], [embeddings[2]])[0][0]
  confidence = abs(sim_a - sim_b)

  # Route based on confidence
  if confidence > 0.30:
      prediction = (sim_a > sim_b)  # Use embedding
  else:
      prediction = symbolic_prediction(anchor, text_a, text_b)  # Use symbolic



## Next Steps

1. **If ensemble improves accuracy** (>71%):
   - Test on sample set to verify generalization
   - Prepare test set predictions using best ensemble
   - Consider upgrading to better LLM (Gemini 2.0 Pro, Claude 3.5)

2. **If ensemble doesn't improve much**:
   - Try Approach #2: Upgrade LLM in symbolic approach
   - Try Approach #6: Error analysis to understand failure modes
   - Consider generating better synthetic data

3. **Error Analysis**:
   - Manually review cases where both models fail
   - Identify patterns (story length, genre, complexity)
   - Develop targeted improvements